In [12]:
from IPython.display import Image
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error as MSE
from sklearn.ensemble import BaggingClassifier , AdaBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

# Hyperparameters

Machine learning models are characterized by parameters and hyperparameters .
- **Parameters** : learned from data
    - **CART** : split points of a node , split features oof a node
- **hyperparameter** : not learned from data set prior to training 
    - **CART** : max_depth , min_samples_leaf , splitting criterion

### What is hyperparameter tuning?
- **Problem** : search for a set of optimal hyperparameters for a learning alogorithm
- **Solution**: find a set of optimal hyperparameters that results in an optimal model
- **Optimal model** : yields an optimal score
- **Score** : In sklearn defaults to accuracy (classification) and R^2 (regression)
- Cross Validation is used to estimate the generalization performance

### Why tune hyperparameters?
- In sklearn a model's default hyperparameters are not optimal for all problems
- Hyperparameters should be tuned to obtain the best model performance

### Grid Search Cross Validation
- Manually set a grid of discrete hyperparameter values
- Set a metric for scoring model performance
- Search exhaustively through the grid
- For each set of hyperparameters , evaluate each model's score 
- the optimal hyperparameters are those of the model achieving the best CV score

### Grid search cross validation : example
- Hyperparameters grids :
    - max_depth = {2,3,4}
    - min_samples_leaf = {0.5,0.1}
- Hyperparameter space = { (2,0.5)......}
- CV scores = {score(2,0.05) , ............}
- Optimal hyperparameters = set of hyperparameters corresponding to the best CV score

In [3]:
dt = DecisionTreeClassifier(random_state=1)

In [4]:
print(dt.get_params())

{'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': 1, 'splitter': 'best'}


In [11]:
df = pd.read_csv(r"../3. Bagging/breast-cancer.csv")

X = df.drop('diagnosis', axis=1).to_numpy()
y_labels = df['diagnosis'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y_labels, test_size=0.3, stratify=y_labels, random_state=1)

params_dt = {
    'max_depth' : [3,4,5,6],
    'min_samples_leaf' : [0.04,0.06,0.08],
    'max_features' : [0.2,0.4,0.6,0.8]
}

grid_dt = GridSearchCV(estimator=dt,param_grid=params_dt,scoring='accuracy',cv=10,n_jobs=-1)

grid_dt.fit(X_train,y_train)

best_hyperparams = grid_dt.best_params_
print("Best hyperparameters:\n" , best_hyperparams)

best_cv_score = grid_dt.best_score_
print('Best CV accuracy'.format(best_cv_score))

best_model = grid_dt.best_estimator_
test_acc = best_model.score(X_test,y_test)

print("Test set accuracy of best model: {:.3f}".format(test_acc))


Best hyperparameters:
 {'max_depth': 3, 'max_features': 0.4, 'min_samples_leaf': 0.06}
Best CV accuracy
Test set accuracy of best model: 0.936


# Random Forest Hyperparameters
- In addition to the hyperparameters of the CARTs forming random forests
- The ensemble itself is characterized by other hyperparameters such as the number of estimators whether it uses bootstraping or not and so on

# Tuning is expensive
- Hyperparameter tuning:
    - computationally expensive
    - sometimes leads to very slight improvement
- It is desired to weigh the impact of tuning on the pipeline of your data analysis project as a whole in order to understand if it is worth pursuing

In [13]:
rf = RandomForestRegressor(random_state=1)
rf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 1,
 'verbose': 0,
 'warm_start': False}

In [ ]:
df_csv = pd.read_csv("../1. Classification and Regression Trees/Regression/output_data.csv")

df_csv = df_csv.drop(["car_name","model_year","origin"],axis=1)

df_csv['horsepower'] = pd.to_numeric(df_csv['horsepower'], errors='coerce')


feature = df_csv.drop("mpg",axis=1).values
target = df_csv['mpg'].values

X_train, X_test, y_train, y_test = train_test_split(feature, target, test_size=0.3, random_state=1)


params_rf = {
    'n_estimators' : [300,400,500],
    'max_depth' : [4,6,8],
    'min_samples_leaf' : [0.1,0.2],
    'max_features' : ['log2','sqrt']
}

grid_rf = GridSearchCV(estimator=rf,param_grid=params_rf,scoring='neg_mean_squared_error',cv=3,n_jobs=-1,verbose=1)

grid_rf.fit(X_train,y_train)


Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best hyperparameters:
 {'max_depth': 4, 'max_features': 'log2', 'min_samples_leaf': 0.1, 'n_estimators': 300}


In [22]:

best_hyperparams = grid_rf.best_params_

print("Best hyperparameters:\n" , best_hyperparams)


Best hyperparameters:
 {'max_depth': 4, 'max_features': 'log2', 'min_samples_leaf': 0.1, 'n_estimators': 300}


In [23]:
best_model = grid_rf.best_estimator_

y_pred = best_model.predict(X_test)

rmse_test = MSE(y_test,y_pred) ** 0.5

print("Test set RMSE of rf : {:.2f}".format(rmse_test))

Test set RMSE of rf : 3.49
